# Notebook 03 — Ratcheting and key compromise

We crank the symmetric ratchet through 10 messages, then simulate a key-compromise at step 5 to see what's safe and what isn't.

In [1]:
from pqmsg.kdf import derive_chain_step
from cryptography.hazmat.primitives.ciphers.aead import ChaCha20Poly1305
import os

## Step 1 — watch the chain advance

In [2]:
chain_key = os.urandom(32)
for i in range(10):
    msg_key, next_ck = derive_chain_step(chain_key)
    aead = ChaCha20Poly1305(msg_key)
    nonce = os.urandom(12)
    ct = aead.encrypt(nonce, f"message {i}".encode(), None)
    print(f"msg {i}: msg_key[:4]={msg_key[:4].hex()}  chain_key[:4]={next_ck[:4].hex()}")
    chain_key = next_ck

msg 0: msg_key[:4]=45252439  chain_key[:4]=16726559
msg 1: msg_key[:4]=86ce8aed  chain_key[:4]=0080be6f
msg 2: msg_key[:4]=902ae063  chain_key[:4]=48267653
msg 3: msg_key[:4]=c4edf71b  chain_key[:4]=10d99792
msg 4: msg_key[:4]=5f575104  chain_key[:4]=e95a2dd7
msg 5: msg_key[:4]=5ec3410c  chain_key[:4]=03788e78
msg 6: msg_key[:4]=79ea3435  chain_key[:4]=14837e00
msg 7: msg_key[:4]=cda55a48  chain_key[:4]=24bf3f43
msg 8: msg_key[:4]=a1fba753  chain_key[:4]=52dde580
msg 9: msg_key[:4]=c76a6f72  chain_key[:4]=4c3820af


## Step 2 — simulate compromise at step 5

Suppose an attacker steals `chain_key` at step 5 (after message 4 has been sent).

In [3]:
chain_key = os.urandom(32)
stolen = None
saved_keys = []
for i in range(10):
    if i == 5:
        stolen = chain_key
    msg_key, next_ck = derive_chain_step(chain_key)
    saved_keys.append(msg_key)
    chain_key = next_ck

print("stolen chain_key (first 8B):", stolen[:8].hex())

stolen chain_key (first 8B): 1c1f633012ac4eaf


## Step 3 — can the attacker decrypt past messages?

From the stolen chain_key, the attacker tries to derive message keys 0..4.

In [4]:
ck = stolen
attacker_keys_future = []
for i in range(5, 10):
    mk, nxt = derive_chain_step(ck)
    attacker_keys_future.append(mk)
    ck = nxt

print("Future messages (5..9): attacker reconstructs every msg_key.")
for i in range(5, 10):
    matches = attacker_keys_future[i - 5] == saved_keys[i]
    print(f"  msg {i} key reconstructed: {matches}")

print("\nPast messages (0..4): attacker has chain_key at step 5, and HKDF is one-way.")
print("  There is NO function that turns next_chain into previous chain_key.")
print("  Attacker CANNOT compute msg_keys 0..4 from `stolen`.")

Future messages (5..9): attacker reconstructs every msg_key.
  msg 5 key reconstructed: True
  msg 6 key reconstructed: True
  msg 7 key reconstructed: True
  msg 8 key reconstructed: True
  msg 9 key reconstructed: True

Past messages (0..4): attacker has chain_key at step 5, and HKDF is one-way.
  There is NO function that turns next_chain into previous chain_key.
  Attacker CANNOT compute msg_keys 0..4 from `stolen`.


## Takeaways

1. **Forward secrecy holds**: messages before the compromise stay confidential because HKDF is one-way.
2. **Backward secrecy fails**: every message from the compromise onward is readable.

The **full Double Ratchet** adds a DH step every few messages — so a fresh X25519/ML-KEM exchange refreshes the chain key with entropy the attacker hasn't seen. That closes the backward-secrecy gap. We deliberately omit it here so the limitation is explicit. In a real messenger (Signal, iMessage PQ3), you always want the full ratchet.